# 4) Approval Time Analysis by Borough and Job Type

In [1]:
import os
import sys
import json
from pathlib import Path


import pandas as pd
import numpy as np
import altair as alt


import geopandas as gpd
from shapely.geometry import Point

# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)
#pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
#pd.set_option("display.max_colwidth", 200)

print("Versions ->",
      "pandas:", pd.__version__,
      "| geopandas:", gpd.__version__)

Versions -> pandas: 2.2.2 | geopandas: 1.1.1


In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
file_path = '/content/drive/MyDrive/CS424-Assignment3/df_sub.pkl'
FINAL_DF = pd.read_pickle(file_path)

In [4]:

from pandas.api.types import is_bool_dtype, is_numeric_dtype

#  Defining work-type columns
worktype_cols = [
    "Sprinkler (Work Type)","Plumbing (Work Type)","Boiler Equipment (Work Type)",
    "Earth Work (Work Type)","Foundation (Work Type)","General Construction (Work Type)",
    "Mechanical Systems (Work Type)","Place of Assembly (Work Type)",
    "Protection Mechanical Methods (Work Type)","Sidewalk Shed (Work Type)",
    "Structural (Work Type)","Support of Excavation (Work Type)",
    "Temporary Place of Assembly (Work Type)",
]

#  handling pandas BooleanDtype cleanly
def coerce_bool_col(col: pd.Series) -> pd.Series:
    if is_bool_dtype(col):
        # Nullable boolean -> fill NA then keep plain bool
        return col.fillna(False).astype(bool)
    if is_numeric_dtype(col):
        return col.fillna(0).astype(int).astype(bool)
    s = col.astype(str).str.strip().str.lower()
    return s.isin({"true","t","yes","y","1"})

# Ensuring columns exist
for c in worktype_cols:
    if c not in FINAL_DF.columns:
        FINAL_DF[c] = False

# Coercing all work-type columns
for c in worktype_cols:
    FINAL_DF[c] = coerce_bool_col(FINAL_DF[c])

# Recomputing complexity score
FINAL_DF["worktype_count"] = FINAL_DF[worktype_cols].sum(axis=1)

# Make sure approval_days exists
for c in ["Filing Date","Approved Date","First Permit Date","Signoff Date"]:
    if c in FINAL_DF.columns:
        FINAL_DF[c] = pd.to_datetime(FINAL_DF[c], errors="coerce")
if "effective_approval_date" not in FINAL_DF.columns:
    eff = FINAL_DF.get("Approved Date", pd.Series(pd.NaT, index=FINAL_DF.index))
    eff = eff.combine_first(FINAL_DF.get("First Permit Date", pd.Series(pd.NaT, index=FINAL_DF.index)))
    eff = eff.combine_first(FINAL_DF.get("Signoff Date", pd.Series(pd.NaT, index=FINAL_DF.index)))
    FINAL_DF["effective_approval_date"] = eff
if "approval_days" not in FINAL_DF.columns:
    FINAL_DF["approval_days"] = (FINAL_DF["effective_approval_date"] - FINAL_DF["Filing Date"]).dt.days

# Rebuilding the minimal tables used by the visualization
events_table = FINAL_DF[[
    "Job Filing Number","Filing Date","effective_approval_date","approval_days",
    "Borough","Job Type","Initial Cost","Total Construction Floor Area",
    "worktype_count", *worktype_cols
]].copy()

# Non-negative approvals only for charts
mask = events_table["approval_days"].notna() & (events_table["approval_days"] >= 0)
events_nonneg = events_table.loc[mask, ["Filing Date","approval_days","Borough","Job Type","Job Filing Number"]].copy()

# Binning and capping
vals = events_nonneg["approval_days"].astype(float).to_numpy()
if vals.size:
    q = np.nanpercentile(vals, [25, 75, 99])
    iqr = float(q[1]-q[0]) if np.isfinite(q).all() else 0.0
    fd = 2*iqr/(vals.size**(1/3)) if vals.size else 14.0
    bin_step = max(7.0, round((fd if np.isfinite(fd) and fd>0 else 14.0)/7.0)*7.0)
    cap_days = int(min((q[2] if np.isfinite(q[2]) else 730), 730))
else:
    bin_step = 14.0
    cap_days = 730

events_nonneg["ym"] = pd.to_datetime(events_nonneg["Filing Date"]).dt.strftime("%Y-%m")
events_nonneg["approval_days_cap"] = events_nonneg["approval_days"].clip(0, cap_days)

monthly_medians_job = (
    events_nonneg.groupby(["ym","Job Type"], dropna=False)["approval_days"]
    .median().rename("median_approval_days").reset_index()
)
monthly_medians_job["n_approved"] = (
    events_nonneg.groupby(["ym","Job Type"], dropna=False)["approval_days"].size().to_numpy()
)

monthly_medians_boro = (
    events_nonneg.groupby(["ym","Borough"], dropna=False)["approval_days"]
    .median().rename("median_approval_days").reset_index()
)
monthly_medians_boro["n_approved"] = (
    events_nonneg.groupby(["ym","Borough"], dropna=False)["approval_days"].size().to_numpy()
)



In [5]:
 # 1) Define the last 12 FULL months by Filing Date
df0 = FINAL_DF[["Filing Date", "Approved Date", "Borough", "Job Type"]].dropna(subset=["Filing Date"]).copy()

data_max = df0["Filing Date"].max()
this_month_start = pd.Timestamp(year=data_max.year, month=data_max.month, day=1)
end_of_this_month = this_month_start + pd.offsets.MonthEnd(1)
last_full_month = this_month_start if data_max >= end_of_this_month else (this_month_start - pd.DateOffset(months=1))
start_12 = last_full_month - pd.DateOffset(months=11)

win = df0[(df0["Filing Date"] >= start_12) & (df0["Filing Date"] <= last_full_month)].copy()

# 2) Keep rows with both Filing Date and Approved Date and compute approval days
win = win.dropna(subset=["Approved Date", "Borough", "Job Type"]).copy()
win["approval_days"] = (win["Approved Date"] - win["Filing Date"]).dt.total_seconds() / 86400.0
win = win[(win["approval_days"] >= 0) & np.isfinite(win["approval_days"])].copy()

# 3) Choose TOP 4 job types (in this 12-month window)
top4_jobs = win["Job Type"].value_counts().head(4).index.tolist()
win = win[win["Job Type"].isin(top4_jobs)].copy()

# 4) Deciding an informative x-extent: p99 of approval_days, but cap at 180 days
p95 = float(np.percentile(win["approval_days"], 95)) if not win.empty else 0.0
p99 = float(np.percentile(win["approval_days"], 99)) if not win.empty else 0.0
LIMIT_DAYS = float(min(max(60.0, round(p99, 0)), 180.0))  # at least 60, at most 180


print(f"Top 4 Job Types: {top4_jobs}")
print(f"Approval days p95≈{p95:.1f}, p99≈{p99:.1f}; using x-extent 0–{LIMIT_DAYS:.0f} days")

# 5) light sampling per group
N_PER = 15000
win_sampled = (
    win.groupby(["Borough", "Job Type"], group_keys=False, observed=False)
       .apply(lambda g: g.sample(min(len(g), N_PER), random_state=42))
       .reset_index(drop=True)
)

# 6) Borough order for consistent faceting
borough_order = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
win_sampled["Borough"] = pd.Categorical(win_sampled["Borough"], categories=borough_order, ordered=True)

Ridgeline window: 2024-10-01 → 2025-09-01
Top 4 Job Types: ['Alteration', 'Alteration CO', 'New Building', 'ALT-CO - New Building with Existing Elements to Remain']
Approval days p95≈77.4, p99≈163.7; using x-extent 0–164 days


/tmp/ipython-input-2203173852.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), N_PER), random_state=42))


####  Final Visualization created after interative process of statistical analysis (per year/trying sampling, etc)

In [ ]:
# Ridgeline (legend filters ONLY ridgeline) ➜ Linked Ranked Job-Type Bar (Top 10)


alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")


DF = FINAL_DF.copy()

for c in ["Filing Date", "Approved Date", "First Permit Date", "Signoff Date"]:
    if c in DF.columns:
        DF[c] = pd.to_datetime(DF[c], errors="coerce")

DF = DF.loc[DF["Filing Date"].notna()].copy()

# Effective approval date: Approved -> First Permit -> Signoff
eff = DF.get("Approved Date", pd.Series(pd.NaT, index=DF.index))
eff = eff.combine_first(DF.get("First Permit Date", pd.Series(pd.NaT, index=DF.index)))
eff = eff.combine_first(DF.get("Signoff Date", pd.Series(pd.NaT, index=DF.index)))
DF["effective_approval_date"] = eff
DF["approval_days"] = (DF["effective_approval_date"] - DF["Filing Date"]).dt.days

E = DF.loc[
    DF["approval_days"].notna() & (DF["approval_days"] >= 0),
    ["Filing Date","approval_days","Borough","Job Type"]
].copy()

# Borough normalization
s = E["Borough"].astype(str).str.strip().str.lower()
mapping = {
    "bronx":"Bronx","brooklyn":"Brooklyn","bk":"Brooklyn",
    "manhattan":"Manhattan","mn":"Manhattan",
    "queens":"Queens","qn":"Queens",
    "staten island":"Staten Island","si":"Staten Island"
}
E["Borough"] = s.map(mapping).fillna(E["Borough"].astype(str).str.strip().str.title())

# Year (string for robust filtering)
E["FilingYear"] = pd.to_datetime(E["Filing Date"]).dt.year
E["YearStr"] = E["FilingYear"].astype("Int64").astype(str)

# Rounded x-cap (p99 up to nearest 30, max 180)
vals = E["approval_days"].astype(float).to_numpy()
p99 = np.nanpercentile(vals, 99) if vals.size else 180
cap_days = int(min(np.ceil(p99/30.0)*30.0, 180))
E["approval_days_cap"] = E["approval_days"].clip(0, cap_days)

borough_order = [b for b in ["Manhattan","Brooklyn","Queens","Bronx","Staten Island"]
                 if b in E["Borough"].dropna().unique().tolist()] or \
                sorted(E["Borough"].dropna().unique().tolist())

#  Controls & selections
year_sel   = alt.param(name="year_sel", value="All",
                       bind=alt.binding_select(options=["All","2021","2022","2023","2024","2025"], name="Year: "))
brush_days = alt.selection_interval(encodings=["x"])

ridge_job_sel = alt.selection_point(fields=["Job Type"], bind="legend", empty="all")

jobtype_order = (["ALT-CO - New Building with Existing", "Alteration", "Alteration CO",
                  "Full Demolition", "New Building", "No Work"]
                 if any(x in E["Job Type"].unique() for x in
                        ["ALT-CO - New Building with Existing","Alteration","Alteration CO",
                         "Full Demolition","New Building","No Work"])
                 else sorted(E["Job Type"].dropna().unique().tolist()))

year_filter_expr = "(year_sel == 'All') || (datum.YearStr == year_sel)"

ridge = (
    alt.Chart(E)
    .add_params(year_sel, brush_days, ridge_job_sel)
    .transform_filter(year_filter_expr)
    .transform_density(
        density="approval_days_cap",
        groupby=["Borough", "Job Type"],
        as_=["approval_days_cap", "density"],
        extent=[0, float(cap_days)],
        steps=200,
        counts=False
    )
    .transform_filter(ridge_job_sel)  # legend affects ONLY the ridgeline
    .mark_area(opacity=0.75)
    .encode(
        x=alt.X("approval_days_cap:Q",
                title=f"Approval time (days, capped at {cap_days})",
                scale=alt.Scale(domain=[0, float(cap_days)])),
        y=alt.Y("density:Q", title=None),
        color=alt.Color(
            "Job Type:N",
            scale=alt.Scale(scheme="tableau10", domain=jobtype_order),
            legend=alt.Legend(
                title="Job Type (click to filter ridgeline)",
                orient="bottom",
                columns=3
            )
        ),
        tooltip=[
            alt.Tooltip("Borough:N"),
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("approval_days_cap:Q", title="Days", format=",.0f"),
            alt.Tooltip("density:Q", title="Density", format=".3f"),
        ],
    )
    .properties(width=900, height=110)
    .facet(
        row=alt.Row("Borough:N",
                    sort=borough_order,
                    title=None,
                    header=alt.Header(labelAngle=0, labelAlign="left"))
    )
    .resolve_scale(y="independent")
    .properties(title="(A) Approval-time ridgeline by Borough — Year filter, legend filters ridgeline, brush over x")
    .interactive()
)

#  (B) Ranked Top-10 Job Types in brushed window

ranked = (
    alt.Chart(E)
    .add_params(year_sel, brush_days)
    .transform_filter(year_filter_expr)
    .transform_filter(brush_days)
    .transform_aggregate(n="count()", groupby=["Job Type"])
    .transform_joinaggregate(total="sum(n)")
    .transform_calculate(share="datum.n / datum.total")
    .transform_window(rank="rank(n)", sort=[alt.SortField("n", order="descending")])
    .transform_filter("datum.rank <= 10")
    .mark_bar()
    .encode(
        y=alt.Y("Job Type:N", sort="-x", title="Job Type"),
        x=alt.X("n:Q", title="Applications in selection"),
        color=alt.Color("Job Type:N", legend=None, scale=alt.Scale(scheme="tableau10")),
        tooltip=[
            alt.Tooltip("Job Type:N"),
            alt.Tooltip("n:Q", title="Count"),
            alt.Tooltip("share:Q", title="Share", format=".1%")
        ]
    )
    .properties(width=900, height=340, title="(B) Top-10 Job Types within the brushed window (count & share)")
)

ridge & ranked
